# Test Qwen2.5-VL-7B pour CassavaVLM

Ce notebook teste le modèle Qwen2.5-VL-7B-Instruct pour valider :
- Le chargement du modèle en FP16
- L'inférence en français
- La consommation de VRAM
- La latence d'inférence
- La capacité à comprendre des images

**Prérequis :**
- GPU avec ≥24GB VRAM (RTX 5090 recommandé)
- Python 3.10 ou 3.11
- Dépendances installées : `pip install -r requirements_minimal.txt`

## Cellule 1: Imports

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image
import torch
import requests
from io import BytesIO
import time
import matplotlib.pyplot as plt

print("✅ Imports réussis")

## Cellule 2: Charger le modèle Qwen2.5-VL-7B

⚠️ **Attention:** Ce téléchargement prend du temps (~14GB) la première fois.

In [ ]:
model_name = "Qwen/Qwen2.5-VL-7B-Instruct"

print("📥 Downloading Qwen2.5-VL-7B...")
print("   (Ceci peut prendre du temps la première fois - ~14GB)")
print()

# Charger le modèle en FP16 (pour économiser VRAM)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",  # Placement automatique sur GPU
    trust_remote_code=True
)

# Charger le processor
processor = AutoProcessor.from_pretrained(
    model_name,
    trust_remote_code=True
)

print("✅ Modèle chargé!")
print(f"   Type: {model.__class__.__name__}")
print(f"   Device: {next(model.parameters()).device}")

## Cellule 3: Vérifier utilisation VRAM

Objectif : VRAM < 20GB pour laisser de la marge pour le fine-tuning

In [ ]:
# Vérifier utilisation mémoire GPU
vram_allocated = torch.cuda.memory_allocated(0) / 1e9
vram_reserved = torch.cuda.memory_reserved(0) / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9

print("💾 Utilisation VRAM:")
print(f"   Allouée:  {vram_allocated:.2f} GB")
print(f"   Réservée: {vram_reserved:.2f} GB")
print(f"   Totale:   {vram_total:.2f} GB")
print(f"   Libre:    {vram_total - vram_reserved:.2f} GB")
print()

# Attendu: ~16-18 GB utilisés (OK pour RTX 5090 32GB)
if vram_allocated < 20:
    print("✅ VRAM OK - Marge suffisante pour le fine-tuning")
else:
    print("⚠️  VRAM élevée - Considérer la quantification 8-bit")

## Cellule 4: Charger une image de test depuis internet

In [ ]:
# Utiliser une image de test depuis internet
image_url = "https://raw.githubusercontent.com/ultralytics/yolov5/master/data/images/zidane.jpg"
response = requests.get(image_url)
image = Image.open(BytesIO(response.content)).convert("RGB")

# Afficher l'image
plt.figure(figsize=(8, 6))
plt.imshow(image)
plt.axis('off')
plt.title("Image de test")
plt.show()

print(f"Taille image: {image.size}")

## Cellule 5: Test d'inférence en ANGLAIS (baseline)

In [ ]:
# Créer le message
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Describe this image in detail."}
        ]
    }
]

# Préparer les inputs
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(
    text=[text],
    images=[image],
    return_tensors="pt"
).to("cuda")

# Générer la réponse
print("🤖 Génération en cours...")
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True
    )

# Décoder la réponse
response = processor.decode(output[0], skip_special_tokens=True)
print("\n" + "="*50)
print("RÉPONSE DU MODÈLE (ANGLAIS):")
print("="*50)
print(response)

## Cellule 6: Test en FRANÇAIS ✨ (VALIDATION CRITIQUE)

C'est le test le plus important pour notre projet cassava français !

In [ ]:
# Test avec question en français
messages_fr = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Décris cette image en détail en français."}
        ]
    }
]

text_fr = processor.apply_chat_template(
    messages_fr,
    tokenize=False,
    add_generation_prompt=True
)

inputs_fr = processor(
    text=[text_fr],
    images=[image],
    return_tensors="pt"
).to("cuda")

print("🇫🇷 Test en français...")
with torch.no_grad():
    output_fr = model.generate(
        **inputs_fr,
        max_new_tokens=200,
        temperature=0.7
    )

response_fr = processor.decode(output_fr[0], skip_special_tokens=True)
print("\n" + "="*50)
print("RÉPONSE EN FRANÇAIS:")
print("="*50)
print(response_fr)
print()

# Vérifier que la réponse est bien en français
french_keywords = ['le', 'la', 'les', 'un', 'une', 'des', 'est', 'sont']
is_french = any(keyword in response_fr.lower() for keyword in french_keywords)

if is_french:
    print("✅ Le modèle répond correctement en français!")
else:
    print("⚠️  La réponse ne semble pas être en français - fine-tuning nécessaire")

## Cellule 7: Mesurer la latence d'inférence

Objectif: < 3s par inférence

In [ ]:
# Mesurer temps d'inférence
num_runs = 5
latencies = []

print("⏱️  Mesure de la latence (5 runs)...\n")

for i in range(num_runs):
    start = time.time()
    
    with torch.no_grad():
        output = model.generate(**inputs_fr, max_new_tokens=100)
    
    latency = time.time() - start
    latencies.append(latency)
    print(f"Run {i+1}: {latency:.2f}s")

avg_latency = sum(latencies) / len(latencies)
min_latency = min(latencies)
max_latency = max(latencies)

print(f"\n⏱️  Latence moyenne: {avg_latency:.2f}s")
print(f"⏱️  Latence min:     {min_latency:.2f}s")
print(f"⏱️  Latence max:     {max_latency:.2f}s")
print()

# Évaluation
if avg_latency < 3:
    print("✅ Latence excellente - objectif atteint!")
elif avg_latency < 5:
    print("⚠️  Latence acceptable mais peut être améliorée")
else:
    print("❌ Latence trop élevée - optimisation nécessaire")

## Cellule 8: Test avec image locale (feuille de manioc)

Si vous avez téléchargé une image de manioc dans `data/samples/`

In [ ]:
# Essayer de charger une image de feuille de manioc
local_image_path = "../data/samples/cassava_leaf_example.jpg"

try:
    local_image = Image.open(local_image_path).convert("RGB")
    
    # Afficher l'image
    plt.figure(figsize=(8, 6))
    plt.imshow(local_image)
    plt.axis('off')
    plt.title("Feuille de manioc")
    plt.show()
    
    # Question spécifique au manioc
    messages_cassava = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": local_image},
                {"type": "text", "text": "Quelle maladie affecte cette feuille de manioc ? Décris les symptômes visibles."}
            ]
        }
    ]
    
    text_cassava = processor.apply_chat_template(
        messages_cassava,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs_cassava = processor(
        text=[text_cassava],
        images=[local_image],
        return_tensors="pt"
    ).to("cuda")
    
    print("🌿 Diagnostic d'une feuille de manioc...\n")
    with torch.no_grad():
        output_cassava = model.generate(
            **inputs_cassava,
            max_new_tokens=300,
            temperature=0.7
        )
    
    response_cassava = processor.decode(output_cassava[0], skip_special_tokens=True)
    print("🌿 DIAGNOSTIC MANIOC:")
    print("="*50)
    print(response_cassava)
    print()
    print("✅ Test avec image de manioc réussi!")
    
except FileNotFoundError:
    print("⚠️  Pas d'image de manioc disponible pour l'instant")
    print()
    print("Pour tester avec une vraie image de manioc:")
    print("1. Téléchargez une image depuis Kaggle ou internet")
    print("2. Placez-la dans: data/samples/cassava_leaf_example.jpg")
    print("3. Relancez cette cellule")
except Exception as e:
    print(f"❌ Erreur: {e}")

## Cellule 9: Résumé des résultats

Vérification des critères de succès du MVP

In [ ]:
print("="*60)
print(" 📊 RÉSUMÉ DES TESTS - CassavaVLM MVP")
print("="*60)
print()
print("Critères de succès:")
print()
print(f"✅ Modèle chargé: {model_name}")
print(f"✅ VRAM utilisée: {vram_allocated:.2f} GB / {vram_total:.2f} GB")
print(f"   {'✅' if vram_allocated < 20 else '⚠️ '} Marge pour fine-tuning: {'OUI' if vram_allocated < 20 else 'LIMITE'}")
print()
print(f"✅ Inférence française: {'FONCTIONNELLE' if is_french else 'À AMÉLIORER'}")
print(f"✅ Latence moyenne: {avg_latency:.2f}s")
print(f"   {'✅' if avg_latency < 3 else '⚠️ '} Objectif (<3s): {'ATTEINT' if avg_latency < 3 else 'NON ATTEINT'}")
print()
print("="*60)
print()
print("🎯 PROCHAINES ÉTAPES RECOMMANDÉES:")
print()
print("Option A: Tester sur vraies images de manioc")
print("  → Télécharger dataset Kaggle Cassava")
print("  → Tester diagnostic zéro-shot")
print()
print("Option B: Configurer LoRA pour fine-tuning")
print("  → Créer notebook 01_lora_setup.ipynb")
print("  → Tester PEFT avec paramètres LoRA (r=16, α=32)")
print()
print("Option C: Infrastructure complète")
print("  → Revenir au plan d'implémentation 40 jours")
print("  → Créer pipeline de données complet")
print()
print("="*60)

## Cellule 10: Libérer la mémoire GPU (optionnel)

Exécutez cette cellule si vous voulez libérer la VRAM

In [ ]:
# Décharger le modèle de la VRAM
del model
del processor
torch.cuda.empty_cache()

print("✅ Mémoire GPU libérée")
print(f"   VRAM libre: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")